# Experiment 2: Fine-tuning MiniLM

Fine-tune MiniLM on relevant job-skill pairs from `train.csv` so related jobs and skills have similar embeddings.

Each job is paired with a relevant skill; other skills in the batch act as negatives.

Use relevant pairs (`label > 0`). Hold out 5% of job titles for validation.

After training, rank all development skills for each job and evaluate with `ranking_metrics.py`.

Set `SMOKE = True` for a quick test (4,000 pairs, 1 epoch), or `False` for full training.

In [1]:
import os, sys
from pathlib import Path

# The notebook is in experiments/; config.py and ranking_metrics.py are in the project root.
if not Path("config.py").exists():
    os.chdir(Path.cwd().parent)
sys.path.insert(0, os.getcwd())
print("Project directory:", os.getcwd())

Project directory: /Users/mac/spj/project/Job-Skill-Matching-Ranking-and-Classification-Using-NLP-Approaches


In [2]:
import ast
import json

import numpy as np
import pandas as pd
from datasets import Dataset
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
    losses,
)
from sentence_transformers.evaluation import InformationRetrievalEvaluator
from sentence_transformers.training_args import BatchSamplers

from config import RANDOM_SEED
from ranking_metrics import evaluate_query, save_results

TRAIN_PATH = Path("data/processed/train.csv")
VAL_DIR = Path("data/raw/TaskB/validation")

# ---------- settings ----------
BASE_MODEL = "all-MiniLM-L6-v2"
OUTPUT_DIR = Path("models/minilm_mnrl")   # Fine-tuned model output
RESULT_NAME = "finetuned_minilm_mnrl"
EPOCHS = 2
BATCH_SIZE = 128
LEARNING_RATE = 2e-5
VAL_FRACTION = 0.05
SMOKE = False   # True = quick test (4,000 pairs, 1 epoch; no model is saved)

/var/folders/fy/dyyb2hps4hbdtlhrx_jx3mtm0000gn/T/ipykernel_81952/1757623798.py:7: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import (
/var/folders/fy/dyyb2hps4hbdtlhrx_jx3mtm0000gn/T/ipykernel_81952/1757623798.py:13: DeprecationWarning: Importing from 'sentence_transformers.evaluation' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.evaluation' instead.
  from sentence_transformers.evaluation import InformationRetrievalEvaluator
/var/folders/fy/dyyb2hps4hbdtlhrx_jx3mtm0000gn/T/ipykernel_81952/1757623798.py:14: DeprecationWarning: Importing from 'sentence_transformers.training_args' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.training_args' instead.
  from sentence_transformers.trainin

## 1. Prepare data
Split relevant job-skill pairs by job title.

In [3]:
train_all = pd.read_csv(TRAIN_PATH)
pos = train_all[train_all["label"] > 0][["job_name", "skill_name"]].drop_duplicates().reset_index(drop=True)
print(f"Relevant pairs: {len(pos)} | Jobs: {pos['job_name'].nunique()} | Skills: {pos['skill_name'].nunique()}")

rng = np.random.RandomState(RANDOM_SEED)
jobs = np.array(sorted(pos["job_name"].unique()))
n_val = max(1, int(len(jobs) * VAL_FRACTION))
val_jobs = set(rng.choice(jobs, size=n_val, replace=False))

val_pairs = pos[pos["job_name"].isin(val_jobs)].reset_index(drop=True)
train_pairs = pos[~pos["job_name"].isin(val_jobs)].reset_index(drop=True)

if SMOKE:
    train_pairs = train_pairs.sample(n=min(4000, len(train_pairs)), random_state=RANDOM_SEED).reset_index(drop=True)
print(f"Train: {len(train_pairs)} pairs | Validation: {len(val_pairs)} pairs ({len(val_jobs)} jobs)")

train_dataset = Dataset.from_dict({
    "anchor": train_pairs["job_name"].astype(str).tolist(),
    "positive": train_pairs["skill_name"].astype(str).tolist(),
})

Relevant pairs: 114686 | Jobs: 3011 | Skills: 13224
Train: 108870 pairs | Validation: 5816 pairs (150 jobs)


## 2. Validate during training
Rank training skills for held-out jobs and track retrieval metrics.

In [4]:
all_skills = sorted(pos["skill_name"].astype(str).unique())
skill_id = {s: f"s{i}" for i, s in enumerate(all_skills)}
val_job_list = sorted(val_jobs)
job_id = {j: f"j{i}" for i, j in enumerate(val_job_list)}

relevant = {}
for r in val_pairs.itertuples(index=False):
    relevant.setdefault(job_id[r.job_name], set()).add(skill_id[str(r.skill_name)])

val_evaluator = InformationRetrievalEvaluator(
    queries={job_id[j]: str(j) for j in val_job_list},
    corpus={skill_id[s]: s for s in all_skills},
    relevant_docs=relevant,
    name="val",
    ndcg_at_k=[10, 100],
    map_at_k=[100],
    mrr_at_k=[10],
    show_progress_bar=False,
)

## 3. Train the model

In [5]:
model = SentenceTransformer(BASE_MODEL)
print("device:", model.device)

epochs = 1 if SMOKE else EPOCHS
loss = losses.MultipleNegativesRankingLoss(model)

args = SentenceTransformerTrainingArguments(
    output_dir="models/checkpoints",
    num_train_epochs=epochs,
    per_device_train_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.1,
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=50,
    seed=RANDOM_SEED,
    report_to="none",
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    loss=loss,
    evaluator=val_evaluator,
)
trainer.train()

if not SMOKE:
    OUTPUT_DIR.parent.mkdir(exist_ok=True)
    model.save_pretrained(str(OUTPUT_DIR))
    print("Model saved:", OUTPUT_DIR)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

The `warmup_ratio` argument is deprecated in Transformers v5+, and will also be removed from Sentence Transformers once support for Transformers v4 is dropped. Since you're using Transformers v5+, please use `warmup_steps` (as a float) to specify the warmup ratio instead.


device: mps:0


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

/Users/mac/spj/project/Job-Skill-Matching-Ranking-and-Classification-Using-NLP-Approaches/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Val Cosine Accuracy@1,Val Cosine Accuracy@3,Val Cosine Accuracy@5,Val Cosine Accuracy@10,Val Cosine Precision@1,Val Cosine Precision@3,Val Cosine Precision@5,Val Cosine Precision@10,Val Cosine Recall@1,Val Cosine Recall@3,Val Cosine Recall@5,Val Cosine Recall@10,Val Cosine Ndcg@10,Val Cosine Ndcg@100,Val Cosine Mrr@10,Val Cosine Map@100
1,2.927031,No log,0.593333,0.793333,0.893333,0.926667,0.593333,0.475556,0.421333,0.343333,0.017382,0.041144,0.060986,0.098894,0.390508,0.356523,0.705730,0.145664
2,2.760810,No log,0.573333,0.813333,0.893333,0.933333,0.573333,0.488889,0.433333,0.352000,0.016684,0.042727,0.061995,0.101877,0.398397,0.371556,0.703175,0.154843


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved: models/minilm_mnrl


## 4. Evaluate on development data
Rank all 9,052 skills for each job using the first skill alias.

In [6]:
queries = pd.read_csv(VAL_DIR / "queries", sep="\t").rename(columns={"jobtitle": "job_name", "q_id": "job_id"})
corpus = pd.read_csv(VAL_DIR / "corpus_elements", sep="\t").rename(columns={"c_id": "skill_id"})
corpus["skill_aliases"] = corpus["skill_aliases"].apply(ast.literal_eval)
corpus["skill_name"] = corpus["skill_aliases"].apply(lambda x: x[0])
qrels = pd.read_csv(VAL_DIR / "qrels.tsv", header=None, sep="\t", names=["job_id", "_", "skill_id", "label"])

skill_index = {sid: i for i, sid in enumerate(corpus["skill_id"])}
grades_by_job = {}
for r in qrels.itertuples(index=False):
    grades_by_job.setdefault(r.job_id, np.zeros(len(corpus)))[skill_index[r.skill_id]] = r.label


def encode_texts(texts):
    unique = sorted(set(texts))
    vecs = model.encode(unique, batch_size=64, normalize_embeddings=True, show_progress_bar=True)
    return dict(zip(unique, vecs))


job_emb = encode_texts(queries["job_name"].tolist())
skill_map = encode_texts(corpus["skill_name"].tolist())
skill_emb = np.vstack([skill_map[s] for s in corpus["skill_name"]])

per_query = []
for row in queries.itertuples(index=False):
    if row.job_id in grades_by_job:
        scores = skill_emb @ job_emb[row.job_name]
        per_query.append(evaluate_query(scores, grades_by_job[row.job_id]))

mean = {k: round(float(np.mean([q[k] for q in per_query])), 4) for k in per_query[0]}
print("dev:", mean)

if SMOKE:
    print("Quick test: results not saved. Set SMOKE = False for full training.")
else:
    save_results(
        RESULT_NAME,
        f"Fine-tuned {BASE_MODEL} on relevant train.csv pairs ({epochs} epochs, batch {BATCH_SIZE}, "
        f"learning rate {LEARNING_RATE}); validation split by job title. Cosine similarity; first skill alias.",
        per_query,
    )

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Batches:   0%|          | 0/140 [00:00<?, ?it/s]

dev: {'ndcg_graded': 0.6744, 'ndcg_binary': 0.7217, 'map_binary': 0.2273, 'mrr_binary': 0.6643, 'p@1': 0.5, 'p@5': 0.4704, 'p@10': 0.4441}
Збережено: results/finetuned_minilm_mnrl.json


## 5. Compare results

In [7]:
rows = []
for p in sorted(Path("results").glob("*.json")):
    d = json.loads(p.read_text(encoding="utf-8"))
    rows.append({"approach": p.stem, **{k: v for k, v in d.items() if k != "description"}})

(pd.DataFrame(rows).sort_values("ndcg_graded", ascending=False).reset_index(drop=True)
 if rows else "No result files found in results/.")

AttributeError: 'list' object has no attribute 'items'